# BODHI Symptom Checker — v4


## Configuration

All tunable parameters. Change behavior here.


In [ ]:

LIKELIHOOD_SCORES = {
    'very_high': 1.0,
    'high':      0.8,
    'medium':    0.6,
    'low':       0.4,
    'rare':      0.2,
    'zero':      0.0
}

LIKELIHOOD_SCORES_COMPRESSED = {
    'very_high': 0.8,
    'high': 0.65,
    'medium': 0.5,
    'low': 0.35,
    'rare': 0.2,
    'zero': 0.0
}

# --- Elimination config ---
ELIM_CONFIG = {
    # YES-side: eliminate conditions not connected to ANY confirmed symptom
    'yes_eliminate_unconnected': True,
    
    # NO-side: eliminate conditions where P(S|C) is in this list
    # (they expected the symptom but patient doesn't have it)
    'no_eliminate_psc_levels': ['very_high', 'high'],
    
    # --- Protection layer (P(C|S)-based) ---
    # Conditions with strong P(C|S) get a counter instead of immediate elimination
    'protection_enabled': True,
    'protection_pcs_levels': ['very_high', 'high'],  # P(C|S) levels that trigger protection
    'protection_increment': 0.2,                      # added to counter per elimination attempt
    'protection_threshold': 0.4,                      # eliminated when counter >= this
    
    # --- Triage protection (plug-and-play, OFF by default) ---
    # Emergency conditions get automatic protection regardless of P(C|S)
    'triage_protection': False,                        # flip to True to enable
    'triage_always_protect': ['emergency'],            # triage levels that get protection
    'triage_protection_increment': 0.2,                # same counter system
}

# --- Ranking config ---
RANKING_CONFIG = {
    'yes_point': 1.0,       # +1 per YES-connected condition
    'no_point': -0.2,       # -0.2 per NO-connected condition
    'demographic_method': 'addition',  # 'addition' or 'multiplication'
}

# --- Questioning config ---
QUESTIONING_CONFIG = {
    'max_questions': 10,
    'top_n_discovered': 20,
    'min_pool_size': 3,          # stop if pool reaches this size
    'min_question_score': 0.0,   # skip questions below this score
    'score_threshold': 10,     # stop if any condition reaches this score
    'question_scoring_method': 'normalized_symptom_score',  # 'ev' or 'normalized_symptom_score'
}

# --- Relation type → human-readable question ---
RELATION_QUESTIONS = {
    'duration_since':    'How long have you had this symptom?',
    'location':          'Where is it located?',
    'severity':          'How severe is it?',
    'onset':             'How did it start?',
    'pain_type':         'What type of pain is it?',
    'characteristic':    'What are the characteristics?',
    'aggravated':        'What makes it worse?',
    'relieved':          'What makes it better?',
    'radiating':         'Does it radiate anywhere?',
    'temporal_pattern':  'What is the pattern over time?',
    'duration_lasts':    'How long does each episode last?',
    'laterality':        'Which side is it on?'
}

print("Configuration loaded.")
print(f"Elimination: protection={'ON' if ELIM_CONFIG['protection_enabled'] else 'OFF'}, "
      f"triage_protection={'ON' if ELIM_CONFIG['triage_protection'] else 'OFF'}")
print(f"Ranking: YES={RANKING_CONFIG['yes_point']:+.1f}, NO={RANKING_CONFIG['no_point']:+.1f}")
print(f"Stop: max_questions={QUESTIONING_CONFIG['max_questions']}, "
      f"min_pool={QUESTIONING_CONFIG['min_pool_size']}, "
      f"score_threshold={QUESTIONING_CONFIG['score_threshold']}")


Configuration loaded.
Elimination: protection=ON, triage_protection=OFF
Ranking: YES=+1.0, NO=-0.2
Stop: max_questions=10, min_pool=3, score_threshold=10


## Load Data

In [21]:
import pandas as pd
import math

nodes_symptom    = pd.read_csv('../csv/nodes_symptom.csv')
edges_present_in = pd.read_csv('../csv/edges_present_in.csv')
nodes_condition  = pd.read_csv('../csv/nodes_condition.csv')

# Normalize casing in BOTH likelihood columns
edges_present_in['likelihood_condition_given_symptom'] = edges_present_in['likelihood_condition_given_symptom'].str.lower()
edges_present_in['likelihood_symptom_given_condition'] = edges_present_in['likelihood_symptom_given_condition'].str.lower()

# Normalize overall_likelihood in nodes_condition
nodes_condition['overall_likelihood'] = nodes_condition['overall_likelihood'].str.lower()

# Pre-compute: which UUIDs have edges
UUIDS_WITH_EDGES = set(edges_present_in['symptom_uuid'].unique())

# Pre-compute: total condition weight for P(C) normalization
# P(C) = raw_weight / TOTAL_CONDITION_WEIGHT, so all P(C) values sum to ~1
TOTAL_CONDITION_WEIGHT = nodes_condition['overall_likelihood'].map(LIKELIHOOD_SCORES).sum()

print(f"Loaded: {len(nodes_symptom)} symptom rows, {len(edges_present_in)} edges, {len(nodes_condition)} conditions")
print(f"Symptom UUIDs with edges: {len(UUIDS_WITH_EDGES)} / {nodes_symptom['uuid'].nunique()} "
      f"({len(UUIDS_WITH_EDGES)/nodes_symptom['uuid'].nunique()*100:.0f}%)")
print(f"Total condition weight (normalizer): {TOTAL_CONDITION_WEIGHT:.1f}")

Loaded: 4037 symptom rows, 10352 edges, 779 conditions
Symptom UUIDs with edges: 2559 / 4037 (63%)
Total condition weight (normalizer): 423.6


## Helper Functions

In [22]:
def get_age_bracket_column(age):
    brackets = [
        (1, 'likelihood_age_0_1'), (5, 'likelihood_age_1_5'),
        (12, 'likelihood_age_6_12'), (18, 'likelihood_age_13_18'),
        (30, 'likelihood_age_19_30'), (45, 'likelihood_age_30_45'),
        (60, 'likelihood_age_45_60'),
    ]
    for upper, col in brackets:
        if age <= upper:
            return col
    return 'likelihood_age_60_plus'

def get_gender_column(gender):
    return 'likelihood_male' if gender.upper() in ('M', 'MALE') else 'likelihood_female'

def get_root_uuid(root_snomed_name):
    row = nodes_symptom[
        (nodes_symptom['root_snomed_name'] == root_snomed_name) &
        (nodes_symptom['name'] == nodes_symptom['root_snomed_name'])
    ]
    return row.iloc[0]['uuid'] if len(row) > 0 else None

def get_condition_name(cid):
    row = nodes_condition[nodes_condition['snomed_id'] == cid]
    return row.iloc[0]['name'] if len(row) > 0 else 'Unknown'

def get_condition_triage(cid):
    row = nodes_condition[nodes_condition['snomed_id'] == cid]
    return row.iloc[0]['triage_level'] if len(row) > 0 else 'unknown'

print("Helper functions defined.")


Helper functions defined.


## Function 1: Symptom Expansion

Takes a root symptom name, walks the knowledge graph to discover related symptoms via shared conditions.


In [23]:
def symptom_expansion(symptom_name):
    # Case-insensitive matching: find the correct casing from the data
    match = nodes_symptom[nodes_symptom['root_snomed_name'].str.lower() == symptom_name.lower()]
    if len(match) == 0:
        print(f"ERROR: Symptom '{symptom_name}' not found")
        return None
    
    # Use the actual casing from the data
    canonical_name = match.iloc[0]['root_snomed_name']
    symptom_group = nodes_symptom[nodes_symptom['root_snomed_name'] == canonical_name].copy()
    
    root_snomed_id = symptom_group.iloc[0]['root_snomed_id']
    all_uuids = symptom_group['uuid'].tolist()
    all_edges = edges_present_in[edges_present_in['symptom_uuid'].isin(all_uuids)].copy()
    condition_ids = set(all_edges['condition_snomed_id'].unique())
    
    # Reverse-engineer: discover other root symptoms through shared conditions
    discovered_roots = {}
    for cond_id in condition_ids:
        cond_edges = edges_present_in[edges_present_in['condition_snomed_id'] == cond_id]
        cond_symptoms = nodes_symptom[nodes_symptom['uuid'].isin(cond_edges['symptom_uuid'])][
            ['root_snomed_id', 'root_snomed_name']
        ].drop_duplicates('root_snomed_id')
        
        for _, row in cond_symptoms.iterrows():
            rid = row['root_snomed_id']
            if rid == root_snomed_id or rid in discovered_roots:
                continue
            discovered_roots[rid] = {
                'root_snomed_id': rid,
                'root_snomed_name': row['root_snomed_name'],
                'linked_conditions': set()
            }
        
        for rid in discovered_roots:
            if cond_id in edges_present_in[
                edges_present_in['symptom_uuid'].isin(
                    nodes_symptom[nodes_symptom['root_snomed_id'] == rid]['uuid']
                )
            ]['condition_snomed_id'].values:
                discovered_roots[rid]['linked_conditions'].add(cond_id)
    
    # Rank by total likelihood sum
    for rid, info in discovered_roots.items():
        root_uuid = get_root_uuid(info['root_snomed_name'])
        if root_uuid is None:
            info['total_likelihood_sum'] = 0
            continue
        root_edges = edges_present_in[
            (edges_present_in['symptom_uuid'] == root_uuid) &
            (edges_present_in['condition_snomed_id'].isin(condition_ids))
        ]
        info['total_likelihood_sum'] = root_edges['likelihood_condition_given_symptom'].map(
            LIKELIHOOD_SCORES
        ).sum()
    
    disc_rows = [{
        'root_snomed_id': info['root_snomed_id'],
        'root_snomed_name': info['root_snomed_name'],
        'num_shared_conditions': len(info['linked_conditions']),
        'total_likelihood_sum': info['total_likelihood_sum']
    } for info in discovered_roots.values()]
    
    discovered_df = pd.DataFrame(disc_rows).sort_values(
        'total_likelihood_sum', ascending=False
    ).reset_index(drop=True) if disc_rows else pd.DataFrame()
    
    print(f"Symptom: {canonical_name} (root_id: {root_snomed_id})")
    print(f"Variants: {len(symptom_group)} | Conditions: {len(condition_ids)} | Discovered symptoms: {len(discovered_df)}")
    
    return {
        'starting_root': {'root_snomed_id': root_snomed_id, 'root_snomed_name': canonical_name},
        'starting_variants_df': symptom_group,
        'condition_ids': condition_ids,
        'discovered_roots_df': discovered_df,
        'all_edges_df': all_edges
    }

print("Function 1 (symptom_expansion) defined.")

Function 1 (symptom_expansion) defined.


## Elimination Engine

Two-sided elimination with protection counter system.

**YES answer:** Eliminate conditions not connected to ANY confirmed symptom.

**NO answer:** Eliminate conditions where P(S|C) is high/very_high (they expected the symptom). Protected if P(C|S) is high/very_high, or if triage protection is enabled for emergency conditions.


In [ ]:
def compute_yes_eliminations(symptom_uuid, pool, confirmed_uuids, protection_counters):
    """
    YES answer: eliminate conditions not connected to ANY confirmed symptom.
    Having a symptom is never evidence against a connected condition, even weakly.
    """
    eliminated = set()
    
    if ELIM_CONFIG['yes_eliminate_unconnected']:
        all_confirmed = set(confirmed_uuids) | {symptom_uuid}
        all_confirmed_edges = edges_present_in[edges_present_in['symptom_uuid'].isin(all_confirmed)]
        connected = set(all_confirmed_edges['condition_snomed_id'].unique()) & pool
        eliminated = pool - connected
    
    return eliminated, protection_counters


def compute_no_eliminations(symptom_uuid, pool, protection_counters):
    """
    NO answer: eliminate conditions that expected this symptom (high/very_high P(S|C)).
    Protected if P(C|S) is high/very_high (strong diagnostic link).
    Protected if triage_protection is ON and condition is emergency.
    Protection uses a counter — eliminated only when counter reaches threshold.
    """
    eliminated = set()
    edges = edges_present_in[
        (edges_present_in['symptom_uuid'] == symptom_uuid) &
        (edges_present_in['condition_snomed_id'].isin(pool))
    ]
    
    for _, edge in edges.iterrows():
        cid = edge['condition_snomed_id']
        psc = edge['likelihood_symptom_given_condition']
        pcs = edge['likelihood_condition_given_symptom']
        
        if psc not in ELIM_CONFIG['no_eliminate_psc_levels']:
            continue
        
        # Check P(C|S) protection
        pcs_protected = (
            ELIM_CONFIG['protection_enabled'] and
            pcs in ELIM_CONFIG['protection_pcs_levels']
        )
        
        # Check triage protection
        triage_protected = False
        if ELIM_CONFIG['triage_protection']:
            triage = nodes_condition[nodes_condition['snomed_id'] == cid]['triage_level'].values
            if len(triage) > 0 and triage[0] in ELIM_CONFIG['triage_always_protect']:
                triage_protected = True
        
        if pcs_protected or triage_protected:
            increment = ELIM_CONFIG['protection_increment']
            if triage_protected and not pcs_protected:
                increment = ELIM_CONFIG['triage_protection_increment']
            counter = protection_counters.get(cid, 0) + increment
            protection_counters[cid] = counter
            if counter >= ELIM_CONFIG['protection_threshold']:
                eliminated.add(cid)
        else:
            eliminated.add(cid)
    
    return eliminated, protection_counters


def compute_question_value(symptom_uuid, pool, confirmed_uuids, protection_counters, elim_config, questioning_config):
    """
    Score a question using Expected Value.
    
    P(S) = sum of P(S|Ci) * P(Ci) for all conditions Ci connected to S in pool
    where P(Ci) = LIKELIHOOD_SCORES[overall_likelihood] / TOTAL_CONDITION_WEIGHT
    
    P(YES) = P(S),  P(NO) = 1 - P(S)
    Expected Elimination = P(YES) * yes_elim + P(NO) * no_elim
    Score = Expected Elimination / pool_size
    
    Returns: (score, yes_elim_count, no_elim_count, connected_count, p_yes)
    """    
    edges = edges_present_in[
        (edges_present_in['symptom_uuid'] == symptom_uuid) &
        (edges_present_in['condition_snomed_id'].isin(pool))
    ]
    connected = set(edges['condition_snomed_id'].unique())
    if not connected:
        return 0.0, 0, 0, 0, 0.0
    
    pool_size = len(pool)
    method = questioning_config.get('question_scoring_method', 'ev')
    
    # ---- Normalized Symptom Score ----
    if method == 'normalized_symptom_score':
    # FIXED score: compute against ALL conditions, not just pool
        all_edges = edges_present_in[
            edges_present_in['symptom_uuid'] == symptom_uuid
        ]
        symptom_score = 0.0
        for _, edge in all_edges.iterrows():
            cid = edge['condition_snomed_id']
            psc_val = LIKELIHOOD_SCORES_COMPRESSED.get(edge['likelihood_symptom_given_condition'], 0)
            cond = nodes_condition[nodes_condition['snomed_id'] == cid]
            if len(cond) > 0:
                pc_val = LIKELIHOOD_SCORES_COMPRESSED.get(cond.iloc[0]['overall_likelihood'], 0)
            else:
                pc_val = 0
            symptom_score += psc_val * pc_val

            '''
            If wanted to calculate the P(S) over only the connedted conditions in the pool, we would do this:
            '''
    # if method == 'normalized_symptom_score':
    #     symptom_score = 0.0
    #     for _, edge in edges.iterrows():
    #         cid = edge['condition_snomed_id']
    #         psc_val = LIKELIHOOD_SCORES_COMPRESSED.get(edge['likelihood_symptom_given_condition'], 0)
    #         cond = nodes_condition[nodes_condition['snomed_id'] == cid]
    #         if len(cond) > 0:
    #             pc_val = LIKELIHOOD_SCORES_COMPRESSED.get(cond.iloc[0]['overall_likelihood'], 0)
    #         else:
    #             pc_val = 0
    #         symptom_score += psc_val * pc_val
        
        # normalized_score = symptom_score / len(connected)
        
        # yes/no elim counts (for logging only, not used in scoring)
        all_conf = set(confirmed_uuids) | {symptom_uuid}
        yes_connected = set(
            edges_present_in[edges_present_in['symptom_uuid'].isin(all_conf)]['condition_snomed_id'].unique()
        ) & pool
        yes_elim = len(pool - yes_connected)
        
        no_elim_set = set()
        temp_prot = dict(protection_counters)
        for _, edge in edges.iterrows():
            cid = edge['condition_snomed_id']
            psc = edge['likelihood_symptom_given_condition']
            pcs = edge['likelihood_condition_given_symptom']
            if psc in elim_config['no_eliminate_psc_levels']:
                pcs_prot = elim_config['protection_enabled'] and pcs in elim_config['protection_pcs_levels']
                triage_prot = False
                if elim_config['triage_protection']:
                    t = nodes_condition[nodes_condition['snomed_id'] == cid]['triage_level'].values
                    if len(t) > 0 and t[0] in elim_config['triage_always_protect']:
                        triage_prot = True
                if pcs_prot or triage_prot:
                    c = temp_prot.get(cid, 0) + elim_config['protection_increment']
                    temp_prot[cid] = c
                    if c >= elim_config['protection_threshold']:
                        no_elim_set.add(cid)
                else:
                    no_elim_set.add(cid)
        no_elim = len(no_elim_set)
        
        p_yes = symptom_score  # raw sum before normalization, for logging
        
        return symptom_score, yes_elim, no_elim, len(connected), p_yes
    
    # ---- Expected Value (original) ----
    else:
        p_yes = 0.0
        for _, edge in edges.iterrows():
            cid = edge['condition_snomed_id']
            psc_val = LIKELIHOOD_SCORES.get(edge['likelihood_symptom_given_condition'], 0)
            cond = nodes_condition[nodes_condition['snomed_id'] == cid]
            if len(cond) > 0:
                pc_raw = LIKELIHOOD_SCORES.get(cond.iloc[0]['overall_likelihood'], 0)
                pc_val = pc_raw / TOTAL_CONDITION_WEIGHT
            else:
                pc_val = 0
            p_yes += psc_val * pc_val
        p_yes = min(p_yes, 1.0)
        p_no = 1.0 - p_yes
        
        all_conf = set(confirmed_uuids) | {symptom_uuid}
        yes_connected = set(
            edges_present_in[edges_present_in['symptom_uuid'].isin(all_conf)]['condition_snomed_id'].unique()
        ) & pool
        yes_elim = len(pool - yes_connected)
        
        no_elim_set = set()
        temp_prot = dict(protection_counters)
        for _, edge in edges.iterrows():
            cid = edge['condition_snomed_id']
            psc = edge['likelihood_symptom_given_condition']
            pcs = edge['likelihood_condition_given_symptom']
            if psc in elim_config['no_eliminate_psc_levels']:
                pcs_prot = elim_config['protection_enabled'] and pcs in elim_config['protection_pcs_levels']
                triage_prot = False
                if elim_config['triage_protection']:
                    t = nodes_condition[nodes_condition['snomed_id'] == cid]['triage_level'].values
                    if len(t) > 0 and t[0] in elim_config['triage_always_protect']:
                        triage_prot = True
                if pcs_prot or triage_prot:
                    c = temp_prot.get(cid, 0) + elim_config['protection_increment']
                    temp_prot[cid] = c
                    if c >= elim_config['protection_threshold']:
                        no_elim_set.add(cid)
                else:
                    no_elim_set.add(cid)
        no_elim = len(no_elim_set)
        
        expected_elim = p_yes * yes_elim + p_no * no_elim
        score = expected_elim / pool_size if pool_size > 0 else 0
        return score, yes_elim, no_elim, len(connected), p_yes


print("Elimination engine defined (Expected Value scoring).")

Elimination engine defined (Expected Value scoring).


## Function 2: Dual-Track Questioning

Picks the best question by elimination power. YES/NO answers update both tracks:
- **Elimination track:** conditions removed from pool
- **Ranking track:** +1/-0.2 points accumulated per condition

Shows connected conditions for each question.


In [56]:
def ask_questions(expansion_result):
    """
    Function 2: Dual-track questioning with elimination + ranking.
    
    Returns: (confirmed_uuids, denied_uuids, surviving_pool,
              condition_points, protection_counters, question_log_df)
    """
    config_q = QUESTIONING_CONFIG
    config_r = RANKING_CONFIG
    
    starting_variants = expansion_result['starting_variants_df']
    root_name = expansion_result['starting_root']['root_snomed_name']
    root_snomed_id = expansion_result['starting_root']['root_snomed_id']
    candidate_pool = set(expansion_result['condition_ids'])
    
    confirmed_uuids = []
    denied_uuids = []
    protection_counters = {}
    condition_points = {cid: 0.0 for cid in candidate_pool}
    question_log = []
    
    print("=" * 70)
    print(f"  DUAL-TRACK QUESTIONING FOR: {root_name}")
    print(f"  Starting pool: {len(candidate_pool)} conditions")
    print("=" * 70)
    
    # Auto-confirm root symptom
    root_row = starting_variants[starting_variants['name'] == starting_variants['root_snomed_name']]
    if len(root_row) > 0:
        root_uuid = root_row.iloc[0]['uuid']
        confirmed_uuids.append(root_uuid)
        # Ranking: +1 to connected conditions
        root_edges = edges_present_in[
            (edges_present_in['symptom_uuid'] == root_uuid) &
            (edges_present_in['condition_snomed_id'].isin(candidate_pool))
        ]
        for _, edge in root_edges.iterrows():
            condition_points[edge['condition_snomed_id']] += config_r['yes_point']
        print(f"  Root confirmed: {root_name}")
    
    # ---- Helper: compute full final score for a condition (used in display and threshold) ----
    def _full_score(cid):
        yn = condition_points.get(cid, 0)
        # P(C|S) from confirmed symptoms
        ce = edges_present_in[
            (edges_present_in['symptom_uuid'].isin(confirmed_uuids)) &
            (edges_present_in['condition_snomed_id'] == cid)
        ]
        pcs = ce['likelihood_condition_given_symptom'].map(LIKELIHOOD_SCORES).sum() if len(ce) > 0 else 0
        cond = nodes_condition[nodes_condition['snomed_id'] == cid]
        a_w = cond[get_age_bracket_column(age_input_global)].values[0] if len(cond) > 0 else 1.0
        g_w = cond[get_gender_column(gender_input_global)].values[0] if len(cond) > 0 else 1.0
        a_w = a_w if pd.notna(a_w) else 1.0
        g_w = g_w if pd.notna(g_w) else 1.0
        return yn + pcs + a_w + g_w
    
    # ---- Build askable question pool ----
    askable = []
    
    # Part A: Variant questions
    variants = starting_variants[
        (starting_variants['relation1_type'].notna()) &
        (starting_variants['uuid'].isin(UUIDS_WITH_EDGES))
    ]
    for relation_type, group in variants.groupby('relation1_type'):
        question_text = RELATION_QUESTIONS.get(relation_type, f"About the {relation_type}?")
        selection_type = group.iloc[0]['grouping1_selection_type']
        options = []
        for _, row in group.iterrows():
            if pd.notna(row.get('child2_name')):
                label = f"{row['child1_name']} > {row['child2_name']}"
            elif pd.notna(row.get('child1_name')):
                label = row['child1_name']
            else:
                label = row['name']
            options.append({'label': label, 'uuid': row['uuid'], 'name': row['name']})
        askable.append({
            'type': 'variant', 'root_name': root_name,
            'relation_type': relation_type, 'question': question_text,
            'selection_type': selection_type, 'options': options,
            'score_uuid': group.iloc[0]['uuid'], 'all_uuids': group['uuid'].tolist()
        })
    
    # Part B: Discovered root symptoms
    discovered = expansion_result['discovered_roots_df'].head(config_q['top_n_discovered'])
    for _, row in discovered.iterrows():
        disc_uuid = get_root_uuid(row['root_snomed_name'])
        if disc_uuid is None or disc_uuid not in UUIDS_WITH_EDGES:
            continue
        askable.append({
            'type': 'discovered', 'root_name': row['root_snomed_name'],
            'relation_type': None,
            'question': f"Do you have '{row['root_snomed_name']}'?",
            'selection_type': None, 'options': None,
            'score_uuid': disc_uuid, 'all_uuids': [disc_uuid]
        })
    
    # ---- Questioning loop ----
    questions_asked = 0
    asked_uuids = set()
    
    while askable and questions_asked < config_q['max_questions']:
        
        # Check stop: pool size
        if len(candidate_pool) <= config_q['min_pool_size']:
            print(f"\n  -> Early stop: pool down to {len(candidate_pool)}")
            break
        
        # Check stop: score threshold
        # ── Option 1: Y/N points only (fast, simple) ──
        if config_q['score_threshold'] is not None:
            max_score = max(condition_points[c] for c in candidate_pool) if candidate_pool else 0
            if max_score >= config_q['score_threshold']:
                print(f"\n  -> Early stop: condition reached Y/N score {max_score:.1f}")
                break

        # ── Option 2: Full final score (Y/N + P(C|S) + age + gender) ──
        # if config_q['score_threshold'] is not None:
        #     max_final = 0
        #     max_final_name = ''
        #     for c in candidate_pool:
        #         full = _full_score(c)
        #         if full > max_final:
        #             max_final = full
        #             cond = nodes_condition[nodes_condition['snomed_id'] == c]
        #             max_final_name = cond['name'].values[0] if len(cond) > 0 else '?'
        #     if max_final >= config_q['score_threshold']:
        #         print(f"\n  -> Early stop: {max_final_name} reached final score {max_final:.1f}")
        #         break
        
        # Score all remaining questions
        scored = []
        for q in askable:
            if q['score_uuid'] in asked_uuids:
                continue
            score, ye, ne, conn, p_yes = compute_question_value(
                q['score_uuid'], candidate_pool, confirmed_uuids, protection_counters, ELIM_CONFIG, QUESTIONING_CONFIG
            )
            if score > config_q['min_question_score'] or ye > 0 or ne > 0:
                scored.append((score, ye, ne, conn, p_yes, q))
        
        if not scored:
            print(f"\n  -> No more valuable questions")
            break
        
        scored.sort(key=lambda x: x[0], reverse=True)
        score, ye, ne, conn, p_yes, q = scored[0]
        asked_uuids.add(q['score_uuid'])
        questions_asked += 1
        
        # Show connected conditions WITH their current full score
        q_edges = edges_present_in[
            (edges_present_in['symptom_uuid'] == q['score_uuid']) &
            (edges_present_in['condition_snomed_id'].isin(candidate_pool))
        ]
        connected_cids = set(q_edges['condition_snomed_id'].unique())
        
        # Build "Name (score)" display for connected conditions
        connected_display = []
        for cid in list(connected_cids)[:5]:
            cname = get_condition_name(cid)[:30]
            cscore = _full_score(cid)
            connected_display.append(f"{cname} ({cscore:.1f})")
        connected_str = ', '.join(connected_display)
        if len(connected_cids) > 5:
            connected_str += '...'
        
        # ---- Ask the question ----
        if q['type'] == 'variant':
            options = q['options']
            options_str = ", ".join(f"{i+1}.{o['label']}" for i, o in enumerate(options))
            mode = "pick ONE, 0=none" if q['selection_type'] == 's' else "pick comma-separated, 0=none"
            
            print(f"\n  Q{questions_asked} [variant] (score:{score:.3f} | P(YES):{p_yes:.3f} | YES_elim:{ye} NO_elim:{ne})")
            print(f"    {q['question']}")
            for i, o in enumerate(options, 1):
                print(f"      {i}. {o['label']}")
            print(f"    Connected to: {connected_str}")
            
            user_input = input(f"{q['root_name']} - {q['question']} ({options_str}) [{mode}]: ").strip()
            
            if user_input in ('0', ''):
                for o in options:
                    denied_uuids.append(o['uuid'])
                # Ranking: -0.2 to connected
                for cid in connected_cids:
                    if cid in condition_points:
                        condition_points[cid] += config_r['no_point']
                # Elimination: NO
                eliminated, protection_counters = compute_no_eliminations(
                    q['score_uuid'], candidate_pool, protection_counters
                )
                candidate_pool -= eliminated
                if eliminated:
                    print(f"    -> Skipped. Eliminated {len(eliminated)}:")
                    for cid in eliminated:
                        print(f"      X {get_condition_name(cid)} ({get_condition_triage(cid)})")
                else:
                    print(f"    -> Skipped.")
            else:
                try:
                    selected_uuids = set()
                    if q['selection_type'] == 's':
                        idx = int(user_input) - 1
                        if 0 <= idx < len(options):
                            confirmed_uuids.append(options[idx]['uuid'])
                            selected_uuids.add(options[idx]['uuid'])
                            print(f"    -> Confirmed: {options[idx]['label']}")
                    else:
                        for idx in (int(x.strip())-1 for x in user_input.split(',')):
                            if 0 <= idx < len(options):
                                confirmed_uuids.append(options[idx]['uuid'])
                                selected_uuids.add(options[idx]['uuid'])
                                print(f"    -> Confirmed: {options[idx]['label']}")
                    for o in options:
                        if o['uuid'] not in selected_uuids:
                            denied_uuids.append(o['uuid'])
                    # Ranking: +1 to connected
                    for cid in connected_cids:
                        if cid in condition_points:
                            condition_points[cid] += config_r['yes_point']
                    # Elimination: YES
                    eliminated, protection_counters = compute_yes_eliminations(
                        q['score_uuid'], candidate_pool, confirmed_uuids, protection_counters
                    )
                    candidate_pool -= eliminated
                    if eliminated:
                        print(f"    Eliminated {len(eliminated)} unconnected conditions")
                except ValueError:
                    print("    -> Invalid input, skipping")
        
        else:
            # Discovered symptom: yes/no
            print(f"\n  Q{questions_asked} [discovered] (score:{score:.3f} | P(YES):{p_yes:.3f} | YES_elim:{ye} NO_elim:{ne})")
            print(f"    Do you have '{q['root_name']}'?")
            print(f"    Connected to: {connected_str}")
            
            answer = input(f"Do you have '{q['root_name']}'? (y/n): ").strip().lower()
            
            if answer in ('y', 'yes'):
                confirmed_uuids.append(q['score_uuid'])
                print(f"    -> Confirmed: {q['root_name']}")
                # Ranking: +1
                for cid in connected_cids:
                    if cid in condition_points:
                        condition_points[cid] += config_r['yes_point']
                # Elimination: YES
                eliminated, protection_counters = compute_yes_eliminations(
                    q['score_uuid'], candidate_pool, confirmed_uuids, protection_counters
                )
            else:
                denied_uuids.append(q['score_uuid'])
                print(f"    -> No")
                # Ranking: -0.2
                for cid in connected_cids:
                    if cid in condition_points:
                        condition_points[cid] += config_r['no_point']
                # Elimination: NO
                eliminated, protection_counters = compute_no_eliminations(
                    q['score_uuid'], candidate_pool, protection_counters
                )
            
            candidate_pool -= eliminated
            if eliminated:
                print(f"    Eliminated {len(eliminated)}:")
                for cid in eliminated:
                    print(f"      X {get_condition_name(cid)} ({get_condition_triage(cid)})")
        
        # Log
        question_log.append({
            'order': questions_asked,
            'type': q['type'],
            'symptom': q['root_name'],
            'relation_type': q['relation_type'] or '-',
            'score': round(score, 3),
            'p_yes': round(p_yes, 3),
            'yes_elim': ye,
            'no_elim': ne,
            'connected_conditions': ', '.join(get_condition_name(c) for c in connected_cids),
            'pool_after': len(candidate_pool)
        })
        
        print(f"    Pool: {len(candidate_pool)} conditions remaining")
    
    # Remaining questions count
    remaining = sum(1 for q in askable if q['score_uuid'] not in asked_uuids)
    
    print(f"\n{'=' * 70}")
    print(f"  Questions asked: {questions_asked} | Remaining possible: {remaining}")
    print(f"  Confirmed: {len(confirmed_uuids)} | Denied: {len(denied_uuids)}")
    print(f"  Surviving conditions: {len(candidate_pool)}")
    print(f"  Eliminated: {len(expansion_result['condition_ids']) - len(candidate_pool)}")
    print(f"{'=' * 70}")
    
    return (confirmed_uuids, denied_uuids, candidate_pool,
            condition_points, protection_counters, pd.DataFrame(question_log))


print("Function 2 (ask_questions) defined.")

Function 2 (ask_questions) defined.


## Function 3: Rank Conditions

Scores surviving conditions using the dual-track system:
- Y/N points from the questioning phase
- P(C|S) likelihood from confirmed symptoms
- Demographics (age + gender)


In [57]:
def rank_conditions(confirmed_uuids, denied_uuids, surviving_pool,
                    condition_points, age, gender):
    """
    Rank surviving conditions.
    
    final_score = Y/N_points + sum(P(C|S) from confirmed) + age_weight + gender_weight
    
    Returns: (result_df, scoring_detail_df)
    """
    if not surviving_pool:
        print("No surviving conditions to rank.")
        return pd.DataFrame(), pd.DataFrame()
    
    # P(C|S) scores from confirmed symptoms
    confirmed_edges = edges_present_in[
        (edges_present_in['symptom_uuid'].isin(confirmed_uuids)) &
        (edges_present_in['condition_snomed_id'].isin(surviving_pool))
    ].copy()
    
    confirmed_edges['pcs_score'] = confirmed_edges[
        'likelihood_condition_given_symptom'
    ].map(LIKELIHOOD_SCORES).fillna(0)
    
    pcs_agg = confirmed_edges.groupby('condition_snomed_id').agg(
        pcs_total=('pcs_score', 'sum'),
        num_symptom_matches=('symptom_uuid', 'nunique')
    ).reset_index()
    
    # Demographics
    gender_col = get_gender_column(gender)
    age_col = get_age_bracket_column(age)
    
    # Build result for all surviving conditions
    rows = []
    for cid in surviving_pool:
        yn = condition_points.get(cid, 0)
        pcs_row = pcs_agg[pcs_agg['condition_snomed_id'] == cid]
        pcs = pcs_row['pcs_total'].values[0] if len(pcs_row) > 0 else 0
        matches = int(pcs_row['num_symptom_matches'].values[0]) if len(pcs_row) > 0 else 0
        
        cond = nodes_condition[nodes_condition['snomed_id'] == cid]
        name = cond['name'].values[0] if len(cond) > 0 else 'Unknown'
        triage = cond['triage_level'].values[0] if len(cond) > 0 else 'unknown'
        type_c = cond['type_condition'].values[0] if len(cond) > 0 else 'unknown'
        
        g_w = cond[gender_col].values[0] if len(cond) > 0 else 1.0
        a_w = cond[age_col].values[0] if len(cond) > 0 else 1.0
        g_w = g_w if pd.notna(g_w) else 1.0
        a_w = a_w if pd.notna(a_w) else 1.0
        
        if RANKING_CONFIG['demographic_method'] == 'multiplication':
            final = yn * pcs * g_w * a_w if pcs > 0 else 0
        else:
            final = yn + pcs + g_w + a_w
        
        rows.append({
            'condition_snomed_id': cid,
            'condition_name': name,
            'triage_level': triage,
            'type_condition': type_c,
            'num_symptom_matches': matches,
            'yn_points': round(yn, 2),
            'pcs_score': round(pcs, 2),
            'gender_weight': round(g_w, 2),
            'age_weight': round(a_w, 2),
            'final_score': round(final, 2)
        })
    
    result = pd.DataFrame(rows).sort_values('final_score', ascending=False).reset_index(drop=True)
    
    # Scoring detail
    detail = confirmed_edges.merge(
        nodes_symptom[['uuid', 'name', 'root_snomed_name']],
        left_on='symptom_uuid', right_on='uuid', how='left'
    ).merge(
        nodes_condition[['snomed_id', 'name']],
        left_on='condition_snomed_id', right_on='snomed_id', how='left',
        suffixes=('_symptom', '_condition')
    )
    detail = detail[[
        'name_symptom', 'root_snomed_name', 'name_condition',
        'likelihood_condition_given_symptom', 'pcs_score'
    ]].copy()
    detail.columns = ['symptom_variant', 'root_symptom', 'condition_name',
                       'likelihood_text', 'pcs_score']
    detail = detail.sort_values(['condition_name', 'pcs_score'], ascending=[True, False]).reset_index(drop=True)
    
    print(f"Ranked {len(result)} surviving conditions")
    print(f"Demographics: gender={gender} ({gender_col}), age={age} ({age_col})")
    print(f"Method: {RANKING_CONFIG['demographic_method']}")
    
    return result, detail


print("Function 3 (rank_conditions) defined.")

Function 3 (rank_conditions) defined.


---

## Test


In [71]:
print("=" * 70)
print("  BODHI SYMPTOM CHECKER v4 — Dual-Track Elimination + Ranking")
print("=" * 70)
print()

symptom_input = input("What symptom are you experiencing? ").strip()
gender_input_global  = input("What is your gender? (M/F): ").strip()
age_input_global     = int(input("What is your age? ").strip())

print(f"\nReceived: symptom='{symptom_input}', gender='{gender_input_global}', age={age_input_global}")

print(f"\n{'=' * 70}")
print("  DONE: SYMPTOM EXPANSION")
print(f"{'=' * 70}")

expansion_result = symptom_expansion(symptom_input)

if expansion_result is None:
    print("Could not process symptom.")
else:

    print(f"\n{'=' * 70}")
    print("  DONE: DUAL-TRACK QUESTIONING")
    print(f"{'=' * 70}")
    
    (confirmed_uuids, denied_uuids, surviving_pool,
     condition_points, protection_counters, question_log) = ask_questions(expansion_result)
    
  
    print(f"\n{'=' * 70}")
    print("  DONE: CONDITION RANKING")
    print(f"{'=' * 70}")
    
    result_df, detail_df = rank_conditions(
        confirmed_uuids, denied_uuids, surviving_pool,
        condition_points, age_input_global, gender_input_global
    )


  BODHI SYMPTOM CHECKER v4 — Dual-Track Elimination + Ranking


Received: symptom='headache', gender='m', age=22

  DONE: SYMPTOM EXPANSION
Symptom: Headache (root_id: 25064002.0)
Variants: 48 | Conditions: 110 | Discovered symptoms: 279

  DONE: DUAL-TRACK QUESTIONING
  DUAL-TRACK QUESTIONING FOR: Headache
  Starting pool: 110 conditions
  Root confirmed: Headache

  Q1 [discovered] (score:36.835 | P(YES):36.835 | YES_elim:1 NO_elim:24)
    Do you have 'Fever'?
    Connected to: Sinusitis (3.6), Botulism (3.0), Viral Exanthem (3.1), Dengue fever (3.4), Puerperal Sepsis (2.4)...
    -> No
    Eliminated 24:
      X Sinusitis (opd_managed)
      X Dengue fever (worrisome)
      X Food poisoning (opd_managed)
      X Cerebral Abscess (emergency)
      X Kyasnur Forest Disease (worrisome)
      X Viral disease (worrisome)
      X Mucormycosis (emergency)
      X Japanese Encephalitis (worrisome)
      X Post vaccination fever (opd_managed)
      X Chikungunya fever (opd_managed)
      X E

### Results

In [72]:
if expansion_result is not None and len(result_df) > 0:
    
    print("=== Question Log ===\n")
    display(question_log)
    
    
    print(f"\n\n=== Top 5 Ranked Conditions ===\n")
    for i, row in result_df.head(5).iterrows():
        print(f"  {i+1}. {row['condition_name']}")
        print(f"     Final: {row['final_score']:.2f} = "
              f"Y/N({row['yn_points']:+.1f}) + P(C|S)({row['pcs_score']:.1f}) + "
              f"Age({row['age_weight']:.1f}) + Gen({row['gender_weight']:.1f})")
        print(f"     Matches: {row['num_symptom_matches']} symptoms | Triage: {row['triage_level']}")
        print()
    
    
    print("\n=== Full Ranking ===")
    display(result_df)
    
    
    confirmed_symptoms = nodes_symptom[nodes_symptom['uuid'].isin(confirmed_uuids)][
        ['uuid', 'root_snomed_name', 'name', 'triage_level']
    ].reset_index(drop=True)
    print(f"\n=== Confirmed Symptoms ({len(confirmed_symptoms)}) ===")
    display(confirmed_symptoms)
    
    
    top5_names = result_df.head(5)['condition_name'].tolist()
    top5_detail = detail_df[detail_df['condition_name'].isin(top5_names)]
    print(f"\n=== Scoring Detail (Top 5) ===")
    display(top5_detail)
    
    
    total_conditions = len(expansion_result['condition_ids'])
    eliminated_count = total_conditions - len(surviving_pool)
    print(f"\n=== Elimination Summary ===")
    print(f"Started with: {total_conditions} conditions")
    print(f"Eliminated:   {eliminated_count}")
    print(f"Survived:     {len(surviving_pool)}")
    print(f"Questions asked: {len(question_log)}")
    
    
    if protection_counters:
        print(f"\n=== Protection Counters ===")
        for cid, counter in sorted(protection_counters.items(), key=lambda x: x[1], reverse=True):
            status = "IN POOL" if cid in surviving_pool else "ELIMINATED"
            print(f"  {get_condition_name(cid):40s} counter={counter:.1f} ({status})")


=== Question Log ===



,order,type,symptom,relation_type,score,p_yes,yes_elim,no_elim,connected_conditions,pool_after
0,1,discovered,Fever,-,36.835,36.835,1,24,"Sinusitis, Botulism, Viral Exanthem, Dengue fe...",86
1,2,discovered,Fatigue,-,33.285,33.285,0,11,"Arachnoid cyst, Craniopharyngioma, Hypoparathy...",75
2,3,discovered,Malaise,-,22.140,22.140,1,2,"Narcolepsy, High altitude sickness, Rabies, Hy...",73
3,4,discovered,Vomit,-,21.755,21.755,1,13,"Arachnoid cyst, Meniere's Disease, Hypertensio...",60
4,5,discovered,Abdominal pain,-,19.337,19.337,1,2,"Hypoparathyroidism, Adrenal insufficiency, Wil...",58
5,6,discovered,Nausea,-,15.873,15.873,1,1,"Neurosyphilis, Cerebral aneurysm, Pheochromocy...",57
6,7,discovered,Cough,-,13.950,13.950,1,0,"Maxillary Sinusitis, Poliomyelitis, Behcets di...",57
7,8,discovered,Muscle weakness,-,12.488,12.488,1,2,"Neurosyphilis, Todd's paralysis, Cervical Spon...",55
8,9,discovered,Dizziness,-,9.938,9.938,1,0,"Neurosyphilis, Cerebral aneurysm, Hypertensive...",55
9,10,discovered,Diarrhea,-,9.740,9.740,1,0,Premenstrual syndrome,55




=== Top 5 Ranked Conditions ===

  1. Iridocyclitis
     Final: 3.60 = Y/N(+1.0) + P(C|S)(0.6) + Age(1.0) + Gen(1.0)
     Matches: 1 symptoms | Triage: worrisome

  2. Astigmatism
     Final: 3.60 = Y/N(+1.0) + P(C|S)(0.6) + Age(1.0) + Gen(1.0)
     Matches: 1 symptoms | Triage: opd_managed

  3. Tension Headache
     Final: 3.55 = Y/N(+1.0) + P(C|S)(0.8) + Age(0.8) + Gen(1.0)
     Matches: 1 symptoms | Triage: opd_managed

  4. Male hypogonadism
     Final: 3.40 = Y/N(+1.0) + P(C|S)(0.4) + Age(1.0) + Gen(1.0)
     Matches: 1 symptoms | Triage: worrisome

  5. Cluster Headache
     Final: 3.40 = Y/N(+1.0) + P(C|S)(0.4) + Age(1.0) + Gen(1.0)
     Matches: 1 symptoms | Triage: opd_managed


=== Full Ranking ===


,condition_snomed_id,condition_name,triage_level,type_condition,num_symptom_matches,yn_points,pcs_score,gender_weight,age_weight,final_score
0,77971008,Iridocyclitis,worrisome,acute_that_may_turn_chronic,1,1.0,0.6,1.00,1.00,3.60
1,82649003,Astigmatism,opd_managed,chronic,1,1.0,0.6,1.00,1.00,3.60
2,398057008,Tension Headache,opd_managed,chronic_with_acute_aggravation,1,1.0,0.8,1.00,0.75,3.55
3,48723006,Male hypogonadism,worrisome,chronic,1,1.0,0.4,1.00,1.00,3.40
4,193031009,Cluster Headache,opd_managed,chronic_with_acute_aggravation,1,1.0,0.4,1.00,1.00,3.40
5,65636009,Keratoconus,opd_managed,chronic,1,1.0,0.4,1.00,1.00,3.40
6,4927003,Acute uveitis,worrisome,acute,1,1.0,0.4,1.00,1.00,3.40
7,444248002,Chronic uveitis,worrisome,chronic,1,1.0,0.4,1.00,1.00,3.40
8,237722004,Empty sella syndrome,worrisome,chronic,1,1.0,0.6,0.75,1.00,3.35
9,78275009,Obstructive sleep apnea,emergency,chronic,1,1.0,0.6,1.00,0.75,3.35



=== Confirmed Symptoms (1) ===


,uuid,root_snomed_name,name,triage_level
0,4720607a-a65c-11eb-8d02-1e003a340630,Headache,Headache,worrisome



=== Scoring Detail (Top 5) ===


,symptom_variant,root_symptom,condition_name,likelihood_text,pcs_score
5,Headache,Headache,Astigmatism,medium,0.6
14,Headache,Headache,Cluster Headache,low,0.4
30,Headache,Headache,Iridocyclitis,medium,0.6
33,Headache,Headache,Male hypogonadism,low,0.4
49,Headache,Headache,Tension Headache,high,0.8



=== Elimination Summary ===
Started with: 110 conditions
Eliminated:   55
Survived:     55
Questions asked: 10

=== Protection Counters ===
  Puerperal Sepsis                         counter=0.2 (ELIMINATED)
  Viral Exanthem                           counter=0.2 (IN POOL)
  Iron deficiency anemia                   counter=0.2 (IN POOL)


In [49]:
nc = nodes_condition[['snomed_id','overall_likelihood']]
lkscoremap = {'very_high': 0.8,
 'high': 0.65,
 'medium': 0.5,
 'low': 0.35,
 'rare': 0.2,
 'zero': 0.0}

nc['overall_likelihood_score'] = nc['overall_likelihood'].apply(lkscoremap.get)

nc2 = edges_present_in.merge(nc,left_on = 'condition_snomed_id',
                        right_on = 'snomed_id',
                         how = 'left')


nc2['likelihood_symptom_given_condition_score'] = nc2['likelihood_symptom_given_condition'].map(lkscoremap.get)

nc2['symptom_score'] = nc2['likelihood_symptom_given_condition_score'] * nc2['overall_likelihood_score']

nc3 = (nc2.groupby(['symptom_uuid'],as_index=False).agg({'symptom_score': 'sum',
                                                   'condition_snomed_id':'nunique'}).sort_values('symptom_score',ascending=False)
 .merge(nodes_symptom[['uuid','name']],left_on = 'symptom_uuid',
        right_on = 'uuid',how = 'left'))


# nc3['symptom_score_normalized'] = nc3['symptom_score'] / nc3['condition_snomed_id']

nc3.sort_values('symptom_score',ascending=False).head(10)

,symptom_uuid,symptom_score,condition_snomed_id,uuid,name
0,47258974-a65c-11eb-8d02-1e003a340630,36.8350,145,47258974-a65c-11eb-8d02-1e003a340630,Fever
1,473914a8-a65c-11eb-8d02-1e003a340630,33.2850,126,473914a8-a65c-11eb-8d02-1e003a340630,Fatigue
2,4720607a-a65c-11eb-8d02-1e003a340630,25.8250,109,4720607a-a65c-11eb-8d02-1e003a340630,Headache
3,4781953e-a65c-11eb-8d02-1e003a340630,22.1400,81,4781953e-a65c-11eb-8d02-1e003a340630,Malaise
4,4737ee0c-a65c-11eb-8d02-1e003a340630,21.7550,89,4737ee0c-a65c-11eb-8d02-1e003a340630,Vomit
5,47172c6c-a65c-11eb-8d02-1e003a340630,19.3375,79,47172c6c-a65c-11eb-8d02-1e003a340630,Abdominal pain
6,47483014-a65c-11eb-8d02-1e003a340630,15.8725,67,47483014-a65c-11eb-8d02-1e003a340630,Nausea
7,471e8e62-a65c-11eb-8d02-1e003a340630,13.9500,48,471e8e62-a65c-11eb-8d02-1e003a340630,Cough
8,4728ddae-a65c-11eb-8d02-1e003a340630,12.8925,51,4728ddae-a65c-11eb-8d02-1e003a340630,Joint pain
9,473ca9f6-a65c-11eb-8d02-1e003a340630,12.7650,57,473ca9f6-a65c-11eb-8d02-1e003a340630,Breathlessness
